# RAG Pipeline — Full Implementation

**Complete, production-grade Retrieval-Augmented Generation system in a single notebook.**
Ready for Google Colab · Kaggle · Local Jupyter

**Features:**
- Document ingestion (PDF / Markdown / TXT)
- Semantic chunking with recursive splitting
- FAISS (or Pinecone) vector store + BM25 sparse retrieval
- Hybrid search with RRF fusion, MMR diversity, and cross-encoder reranking
- Intent classification (8 categories) with per-intent strategies
- Query rewriting and multi-query expansion
- Context compression
- Agentic reasoning with streaming LLM (Ollama / HF API / local)
- NLI-based faithfulness verification
- Structured output validation
- **RAG Evaluation** — synthetic QA generation, LLM-as-a-judge, config sweeping


In [ ]:
import os, sys, json, re, hashlib, time, math, textwrap, warnings, pickle, random
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any
from collections.abc import Generator
from collections import OrderedDict
from threading import Thread

import numpy as np
import httpx

warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)
print(f"Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/rag_pipeline")
elif IS_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
INDEX_DIR = PROJECT_ROOT / "index"
MODELS_DIR = PROJECT_ROOT / "models"
EVAL_DIR = PROJECT_ROOT / "evaluation_output"
for d in [DATA_DIR, INDEX_DIR, MODELS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## 2. Install Dependencies


In [ ]:
def ensure_deps():
    required = [
        "torch", "transformers", "sentence-transformers", "faiss-cpu",
        "numpy", "PyMuPDF", "tqdm", "matplotlib", "pandas", "datasets",
        "huggingface-hub", "python-dotenv", "httpx", "scikit-learn",
    ]
    for pkg in required:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_deps()

try:
    import pymupdf as fitz
except ImportError:
    import fitz

try:
    import faiss
except ImportError:
    faiss = None
    print("WARNING: faiss not installed. VectorStore requires it.")

try:
    import torch
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer,
        AutoModelForSequenceClassification, TextIteratorStreamer,
    )
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
    print("WARNING: transformers not installed. Local models unavailable.")

try:
    from huggingface_hub import InferenceClient
    HAS_HF_HUB = True
except ImportError:
    HAS_HF_HUB = False

try:
    import pinecone
    HAS_PINECONE = True
except ImportError:
    HAS_PINECONE = False

try:
    from rank_bm25 import BM25Okapi
    HAS_BM25 = False  # we use simple TF-IDF instead
except ImportError:
    HAS_BM25 = False

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from datasets import Dataset as HFDataset
except ImportError:
    HFDataset = None

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError:
    TfidfVectorizer = None

print("All imports loaded.")


## 3. Configuration


In [ ]:
CONFIG = {
    # ── Provider Selection ──────────────────────────────────────────
    "LLM_PROVIDER": "HUGGINGFACE_API",        # OLLAMA | HUGGINGFACE_API | HUGGINGFACE
    "EMBEDDING_PROVIDER": "HUGGINGFACE_API",  # LOCAL | HUGGINGFACE_API
    "VECTOR_STORE": "FAISS",                  # FAISS | PINECONE
    "RERANK_PROVIDER": "LOCAL",               # LOCAL | HUGGINGFACE_API
    "VERIFICATION_PROVIDER": "LOCAL",         # LOCAL | HUGGINGFACE_API
    "SPARSE_RETRIEVAL": False,
    "OCR_CORRECTIONS_ENABLED": False,
    "QUERY_REWRITE_ENABLED": False,

    # ── HuggingFace Inference API ───────────────────────────────────
    "HF_TOKEN": "",
    "HF_ROUTER_URL": "https://router.huggingface.co/v1",
    "HF_API_BASE": "https://api-inference.huggingface.co",
    "HF_CHAT_MODEL_ID": "meta-llama/Llama-3.1-8B-Instruct",
    "HF_EMBEDDING_MODEL_ID": "sentence-transformers/all-MiniLM-L6-v2",
    "HF_RERANKER_MODEL_ID": "BAAI/bge-reranker-base",
    "HF_VERIFIER_MODEL_ID": "meta-llama/Llama-3.1-8B-Instruct",
    "HF_TIMEOUT": 60,

    # ── Ollama ──────────────────────────────────────────────────────
    "OLLAMA_BASE_URL": "http://localhost:11434",
    "OLLAMA_MODEL": "llama3.2:3b",

    # ── Pinecone ────────────────────────────────────────────────────
    "PINECONE_API_KEY": "",
    "PINECONE_INDEX": "ragapp",
    "PINECONE_NAMESPACE": "default",
    "PINECONE_CLOUD": "aws",
    "PINECONE_REGION": "us-east-1",

    # ── Embedding ───────────────────────────────────────────────────
    "EMBEDDING_DIMENSION": 384,

    # ── Chunking ────────────────────────────────────────────────────
    "CHUNK_SIZE": 1024,
    "CHUNK_OVERLAP": 150,
    "CHUNK_SEPARATORS": ["\n\n", "\n", ". ", " "],

    # ── Retrieval ───────────────────────────────────────────────────
    "RETRIEVAL_TOP_K": 20,
    "RERANK_TOP_K": 5,
    "RRF_K": 60,
    "RETRIEVAL_NO_ANSWER_THRESHOLD": 0.1,

    # ── MMR ─────────────────────────────────────────────────────────
    "MMR_LAMBDA": 0.5,
    "MMR_ENABLED": True,

    # ── Context Compression ─────────────────────────────────────────
    "COMPRESSION_RATIO": 0.7,
    "MIN_CONTEXT_CHUNKS": 3,

    # ── LLM Generation ──────────────────────────────────────────────
    "LLM_MAX_TOKENS": 1024,
    "LLM_MAX_TOKENS_COMPREHENSIVE": 2048,
    "LLM_TEMPERATURE": 0.1,

    # ── NLI / Faithfulness ──────────────────────────────────────────
    "NLI_MODEL": "microsoft/deberta-v3-base-mnli",
    "NLI_ENTAILMENT_THRESHOLD": 0.70,
    "FAITHFULNESS_THRESHOLD": 0.25,

    # ── System Prompt ───────────────────────────────────────────────
    "SYSTEM_PROMPT": (
        "You are an expert document analyst and research assistant. "
        "Provide COMPREHENSIVE, WELL-STRUCTURED, and DETAILED answers "
        "based ONLY on the provided document context.\n\n"
        "CRITICAL RULES:\n"
        "1. Answer ONLY using information from the provided context. NEVER use outside knowledge.\n"
        "2. If the answer is not in the context, say: 'Not found in document.'\n"
        "3. Be THOROUGH and COMPREHENSIVE.\n"
        "4. Use MARKDOWN FORMATTING: headings, bullet points, numbered lists.\n"
        "5. Cite sources inline using [1], [2], etc.\n"
        "6. Structure your answer: direct answer first, then details, then summary.\n"
        "7. Do NOT use emojis."
    ),
}

print("Configuration loaded.")
print(f"LLM: {CONFIG['LLM_PROVIDER']}  |  Embeddings: {CONFIG['EMBEDDING_PROVIDER']}  |  Vector Store: {CONFIG['VECTOR_STORE']}")
if CONFIG.get("HF_TOKEN"):
    print("HF_TOKEN: configured")
else:
    print("HF_TOKEN: NOT configured (set in CONFIG above for HF API)")


## 4. Data Models


In [ ]:
@dataclass
class DocumentChunk:
    text: str
    source_file: str
    page_number: int
    chunk_index: int
    doc_id: str

@dataclass
class DocumentSection:
    text: str
    source_file: str
    page_or_section: int
    raw_text: Optional[str] = None

@dataclass
class SearchResult:
    chunk: DocumentChunk
    score: float
    global_index: int

@dataclass
class Citation:
    index: int
    source_file: str
    page_number: int
    chunk_index: int
    text_snippet: str
    relevance_score: float

@dataclass
class DocumentInfo:
    filename: str
    file_hash: str
    chunk_count: int

@dataclass
class SourceReference:
    source_file: str; page_number: int; chunk_index: int
    text_snippet: str; relevance_score: float

@dataclass
class ExtractionResult:
    items: list[dict[str, Any]] = field(default_factory=list)
    total_count: int = 0; limit_applied: Optional[int] = None
    sources: list[SourceReference] = field(default_factory=list)
    confidence: float = 0.0; not_found_reason: Optional[str] = None

@dataclass
class CountingResult:
    count: int = 0; entity: str = ""
    sources: list[SourceReference] = field(default_factory=list)
    confidence: float = 0.0; verification_status: str = "verified"

@dataclass
class VerificationResult:
    claim: str = ""; is_true: bool = False; evidence: str = ""
    sources: list[SourceReference] = field(default_factory=list)
    confidence: float = 0.0

@dataclass
class ComparisonResult:
    entities: list[str] = field(default_factory=list)
    comparison_points: list[dict[str, Any]] = field(default_factory=list)
    sources: list[SourceReference] = field(default_factory=list)
    confidence: float = 0.0

@dataclass
class QAResult:
    answer: str = ""; sources: list[SourceReference] = field(default_factory=list)
    confidence: float = 0.0; is_grounded: bool = True

@dataclass
class TableExtractionResult:
    headers: list[str] = field(default_factory=list)
    rows: list[list[str]] = field(default_factory=list)
    source_page: Optional[int] = None; confidence: float = 0.0

def create_not_found_response(intent: str, query: str) -> dict:
    base = {"answer": "Not found in document", "sources": [], "confidence": 0, "is_grounded": True}
    if intent == "extraction":
        return {"items": [], "total_count": 0, "sources": [], "confidence": 0,
                "not_found_reason": f"No information matching '{query}' was found."}
    elif intent == "counting":
        return {"count": 0, "entity": query, "sources": [], "confidence": 0, "verification_status": "not_found"}
    elif intent == "verification":
        return {"claim": query, "is_true": False, "evidence": "No supporting evidence found.",
                "sources": [], "confidence": 0}
    return base

def validate_structured_output(data: dict, intent: str) -> dict:
    try:
        if intent == "extraction": return ExtractionResult(**data).__dict__
        elif intent == "counting": return CountingResult(**data).__dict__
        elif intent == "verification": return VerificationResult(**data).__dict__
        elif intent == "comparison": return ComparisonResult(**data).__dict__
        elif intent == "qa": return QAResult(**data).__dict__
        elif intent == "table_parsing": return TableExtractionResult(**data).__dict__
        return data
    except Exception as e:
        return {**data, "_validation_warning": str(e)}

print("Data models ready.")


## 5. Document Parsing


In [ ]:
_OCR_CURRENCY_PATTERNS = [
    (re.compile(r'\bI(\d+[kK]?)\b'), r'\u20b9\1'),
    (re.compile(r'\bl(\d+[kK]?)\b'), r'\u20b9\1'),
    (re.compile(r'\b1(\d{3,})\b'), r'\u20b9\1'),
    (re.compile(r'Rs\.?\s*(\d+)'), r'\u20b9\1'),
    (re.compile(r'INR\s*(\d+)'), r'\u20b9\1'),
]

def normalize_ocr_artifacts(text: str) -> tuple[str, bool]:
    if not CONFIG["OCR_CORRECTIONS_ENABLED"]:
        return text, False
    had = False
    for pat, repl in _OCR_CURRENCY_PATTERNS:
        if pat.search(text):
            text = pat.sub(repl, text); had = True
    return text, had

def parse_pdf(file_path: Path) -> list[DocumentSection]:
    sections, corrections = [], 0
    doc = fitz.open(str(file_path))
    for page_num in range(len(doc)):
        text = doc[page_num].get_text("text", sort=True).strip()
        if text:
            cleaned, had = normalize_ocr_artifacts(text)
            if had: corrections += 1
            sections.append(DocumentSection(text=cleaned, source_file=file_path.name,
                                            page_or_section=page_num + 1,
                                            raw_text=text if cleaned != text else None))
    doc.close()
    return sections

def parse_markdown(file_path: Path) -> list[DocumentSection]:
    try: raw = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError: raw = file_path.read_text(encoding="latin-1")
    cleaned, _ = normalize_ocr_artifacts(raw)
    sections, sec_num, current = [], 1, []
    for line in cleaned.split("\n"):
        if line.strip().startswith("#") and current:
            text = "\n".join(current).strip()
            if text:
                sections.append(DocumentSection(text=text, source_file=file_path.name,
                                                page_or_section=sec_num)); sec_num += 1
            current = [line]
        else: current.append(line)
    if current:
        text = "\n".join(current).strip()
        if text: sections.append(DocumentSection(text=text, source_file=file_path.name,
                                                page_or_section=sec_num))
    return sections

def parse_text(file_path: Path) -> list[DocumentSection]:
    try: raw = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError: raw = file_path.read_text(encoding="latin-1")
    cleaned, _ = normalize_ocr_artifacts(raw)
    sections, sec_num = [], 1
    for para in cleaned.split("\n\n"):
        text = para.strip()
        if text:
            sections.append(DocumentSection(text=text, source_file=file_path.name,
                                            page_or_section=sec_num)); sec_num += 1
    return sections

def parse_document(file_path: Path) -> list[DocumentSection]:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf": return parse_pdf(file_path)
    elif suffix == ".md": return parse_markdown(file_path)
    elif suffix == ".txt": return parse_text(file_path)
    else: raise ValueError(f"Unsupported file type: {suffix}")

print("Document parser ready.")


## 6. Semantic Chunking


In [ ]:
def _recursive_split(text: str, separators: list[str], chunk_size: int) -> list[str]:
    if len(text) <= chunk_size:
        return [text] if text.strip() else []
    chosen = separators[-1]
    for sep in separators:
        if sep in text: chosen = sep; break
    parts = text.split(chosen)
    chunks, current = [], ""
    for part in parts:
        candidate = current + chosen + part if current else part
        if len(candidate) <= chunk_size: current = candidate
        else:
            if current.strip(): chunks.append(current.strip())
            if len(part) > chunk_size:
                remaining = separators[separators.index(chosen) + 1:]
                if remaining: chunks.extend(_recursive_split(part, remaining, chunk_size))
                else:
                    for i in range(0, len(part), chunk_size):
                        piece = part[i:i+chunk_size].strip()
                        if piece: chunks.append(piece)
                current = ""
            else: current = part
    if current.strip(): chunks.append(current.strip())
    return chunks

def _apply_overlap(chunks: list[str], overlap: int) -> list[str]:
    if overlap <= 0 or len(chunks) <= 1: return chunks
    result = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_tail = chunks[i-1][-overlap:]
        if not chunks[i].startswith(prev_tail):
            result.append(prev_tail + " " + chunks[i])
        else: result.append(chunks[i])
    return result

def chunk_sections(sections: list[DocumentSection], doc_id: str,
                   chunk_size: int = None, chunk_overlap: int = None,
                   separators: list[str] = None) -> list[DocumentChunk]:
    chunk_size = chunk_size or CONFIG["CHUNK_SIZE"]
    chunk_overlap = chunk_overlap or CONFIG["CHUNK_OVERLAP"]
    separators = separators or CONFIG["CHUNK_SEPARATORS"]
    all_chunks, idx = [], 0
    for sec in sections:
        raw = _recursive_split(sec.text, separators, chunk_size)
        overlapped = _apply_overlap(raw, chunk_overlap)
        for text in overlapped:
            if text.strip():
                all_chunks.append(DocumentChunk(text=text, source_file=sec.source_file,
                                                page_number=sec.page_or_section,
                                                chunk_index=idx, doc_id=doc_id)); idx += 1
    return all_chunks

print("Chunker ready.")


## 7. Embeddings


In [ ]:
_embed_model = None
_HF_INFERENCE_CLIENT = None

def get_embed_model():
    global _embed_model
    if CONFIG["EMBEDDING_PROVIDER"] == "HUGGINGFACE_API": return None
    if _embed_model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _embed_model = SentenceTransformer(CONFIG["HF_EMBEDDING_MODEL_ID"], device=device)
    return _embed_model

def _get_hf_client():
    global _HF_INFERENCE_CLIENT
    if _HF_INFERENCE_CLIENT is None:
        _HF_INFERENCE_CLIENT = InferenceClient(token=CONFIG["HF_TOKEN"])
    return _HF_INFERENCE_CLIENT

def embed_texts(texts: list[str], normalize: bool = True) -> np.ndarray:
    if CONFIG["EMBEDDING_PROVIDER"] == "HUGGINGFACE_API":
        model_id = CONFIG["HF_EMBEDDING_MODEL_ID"]
        client = _get_hf_client()
        batch_size = 32
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            result = client.feature_extraction(batch, model=model_id)
            all_embs.append(np.array(result, dtype=np.float32))
        embeddings = np.vstack(all_embs)
        if embeddings.ndim == 3: embeddings = embeddings.mean(axis=1)
        if embeddings.ndim == 1: embeddings = embeddings.reshape(1, -1)
    else:
        model = get_embed_model()
        embeddings = model.encode(texts, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    if normalize:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1
        embeddings = embeddings / norms
    return embeddings

def encode_query(query: str, normalize: bool = True) -> np.ndarray:
    return embed_texts([query], normalize=normalize)

print("Embeddings ready.")


## 8. Vector Stores
### 8a. FAISS Vector Store


In [ ]:
def create_vector_store():
    if CONFIG["VECTOR_STORE"] == "PINECONE":
        return PineconeVectorStore()
    return FAISSVectorStore()

class FAISSVectorStore:
    def __init__(self):
        self.index = faiss.IndexFlatIP(CONFIG["EMBEDDING_DIMENSION"])
        self.metadata: dict[int, DocumentChunk] = {}
        self.doc_index: dict[str, list[int]] = {}
        self.doc_filenames: dict[str, str] = {}

    @property
    def total_chunks(self): return self.index.ntotal
    @property
    def total_documents(self): return len(self.doc_index)

    def add(self, embeddings: np.ndarray, chunks: list[DocumentChunk], doc_id: str):
        start = self.index.ntotal
        self.index.add(embeddings)
        positions = []
        for i, c in enumerate(chunks):
            pos = start + i; self.metadata[pos] = c; positions.append(pos)
        self.doc_index[doc_id] = positions
        if chunks: self.doc_filenames[doc_id] = chunks[0].source_file

    def remove_document(self, doc_id: str) -> bool:
        if doc_id not in self.doc_index: return False
        to_remove = set(self.doc_index[doc_id])
        new_idx = faiss.IndexFlatIP(CONFIG["EMBEDDING_DIMENSION"])
        new_meta, new_doc_idx, new_pos = {}, {}, 0
        for old_pos in range(self.index.ntotal):
            if old_pos in to_remove: continue
            vec = self.index.reconstruct(old_pos).reshape(1, -1)
            new_idx.add(vec)
            chunk = self.metadata[old_pos]
            new_meta[new_pos] = chunk
            new_doc_idx.setdefault(chunk.doc_id, []).append(new_pos)
            new_pos += 1
        self.index = new_idx
        self.metadata = new_meta
        self.doc_index = new_doc_idx
        self.doc_filenames.pop(doc_id, None)
        return True

    def search(self, query_embedding: np.ndarray, top_k: int = 20) -> list[SearchResult]:
        if self.index.ntotal == 0: return []
        k = min(top_k, self.index.ntotal)
        scores, indices = self.index.search(query_embedding, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1: continue
            chunk = self.metadata.get(int(idx))
            if chunk: results.append(SearchResult(chunk=chunk, score=float(score), global_index=int(idx)))
        return results

    def get_document_info(self) -> dict[str, dict]:
        info = {}
        for doc_id, positions in self.doc_index.items():
            info[doc_id] = {"filename": self.doc_filenames.get(doc_id, "unknown"),
                            "file_hash": doc_id, "chunk_count": len(positions)}
        return info

    def has_document(self, doc_id: str) -> bool: return doc_id in self.doc_index

    def get_doc_id_by_filename(self, filename: str) -> Optional[str]:
        for doc_id, name in self.doc_filenames.items():
            if name == filename: return doc_id
        return None

    def save(self):
        faiss.write_index(self.index, str(INDEX_DIR / "faiss.index"))
        with open(INDEX_DIR / "metadata.pkl", "wb") as f:
            pickle.dump({"metadata": self.metadata, "doc_index": self.doc_index,
                         "doc_filenames": self.doc_filenames}, f)

    def load(self) -> bool:
        index_file = INDEX_DIR / "faiss.index"
        meta_file = INDEX_DIR / "metadata.pkl"
        if not index_file.exists() or not meta_file.exists(): return False
        self.index = faiss.read_index(str(index_file))
        with open(meta_file, "rb") as f:
            data = pickle.load(f)
        self.metadata = data["metadata"]
        self.doc_index = data["doc_index"]
        self.doc_filenames = data["doc_filenames"]
        return True

print("FAISSVectorStore ready.")


### 8b. BM25 Sparse Store


In [ ]:
def _tokenize(text: str) -> list[str]:
    return re.findall(r'\b\w+\b', text.lower())

class BM25Store:
    def __init__(self):
        self.corpus: list[list[str]] = []
        self.chunks: list[DocumentChunk] = []
        self.doc_index: dict[str, list[int]] = {}
        self.bm25 = None

    @property
    def total_chunks(self): return len(self.corpus)

    def _rebuild(self):
        if self.corpus:
            from rank_bm25 import BM25Okapi
            self.bm25 = BM25Okapi(self.corpus)
        else: self.bm25 = None

    def add(self, chunks: list[DocumentChunk], doc_id: str):
        start = len(self.corpus)
        for i, c in enumerate(chunks):
            self.corpus.append(_tokenize(c.text))
            self.chunks.append(c)
            self.doc_index.setdefault(doc_id, []).append(start + i)
        self._rebuild()

    def remove_document(self, doc_id: str) -> bool:
        if doc_id not in self.doc_index: return False
        to_remove = set(self.doc_index[doc_id])
        new_corpus, new_chunks, new_idx = [], [], {}
        for old_pos in range(len(self.corpus)):
            if old_pos in to_remove: continue
            new_pos = len(new_corpus)
            new_corpus.append(self.corpus[old_pos])
            new_chunks.append(self.chunks[old_pos])
            cid = self.chunks[old_pos].doc_id
            new_idx.setdefault(cid, []).append(new_pos)
        self.corpus, self.chunks, self.doc_index = new_corpus, new_chunks, new_idx
        self._rebuild()
        return True

    def search(self, query: str, top_k: int = 20) -> list[tuple[int, float, DocumentChunk]]:
        if self.bm25 is None or not self.corpus: return []
        tokens = _tokenize(query)
        scores = self.bm25.get_scores(tokens)
        k = min(top_k, len(scores))
        indices = scores.argsort()[-k:][::-1]
        return [(int(i), float(scores[i]), self.chunks[i]) for i in indices if scores[i] > 0]

    def save(self):
        with open(INDEX_DIR / "bm25.pkl", "wb") as f:
            pickle.dump({"corpus": self.corpus,
                         "chunks": [asdict(c) for c in self.chunks],
                         "doc_index": self.doc_index}, f)

    def load(self) -> bool:
        bm25_file = INDEX_DIR / "bm25.pkl"
        if not bm25_file.exists(): return False
        with open(bm25_file, "rb") as f:
            data = pickle.load(f)
        self.corpus = data["corpus"]
        self.chunks = [DocumentChunk(**c) for c in data["chunks"]]
        self.doc_index = data["doc_index"]
        self._rebuild()
        return True

print("BM25Store ready.")


### 8c. Pinecone Vector Store (optional)


In [ ]:
class PineconeVectorStore:
    def __init__(self):
        if not HAS_PINECONE: raise ImportError("pinecone not installed")
        api_key = CONFIG["PINECONE_API_KEY"]
        if not api_key: raise ValueError("PINECONE_API_KEY not set")
        pinecone.init(api_key=api_key)
        index_name = CONFIG["PINECONE_INDEX"]
        if index_name not in pinecone.list_indexes():
            pinecone.create_index(name=index_name, dimension=CONFIG["EMBEDDING_DIMENSION"],
                                  metric="cosine", spec={"serverless": {"cloud": CONFIG["PINECONE_CLOUD"],
                                                                         "region": CONFIG["PINECONE_REGION"]}})
        self.index = pinecone.Index(index_name)
        self._doc_filenames: dict[str, str] = {}
        self._doc_chunk_counts: dict[str, int] = {}

    @property
    def total_chunks(self): return self.index.describe_index_stats()["total_vector_count"]
    @property
    def total_documents(self): return len(self._doc_filenames)

    def add(self, embeddings: np.ndarray, chunks: list[DocumentChunk], doc_id: str):
        vectors = []
        for i, (emb, chunk) in enumerate(zip(embeddings, chunks)):
            vectors.append((f"{doc_id}_{chunk.chunk_index}", emb.tolist(),
                            {"doc_id": doc_id, "filename": chunk.source_file,
                             "page_number": chunk.page_number, "chunk_index": chunk.chunk_index,
                             "text": chunk.text[:5000]}))
        for i in range(0, len(vectors), 100):
            self.index.upsert(vectors[i:i+100], namespace=CONFIG["PINECONE_NAMESPACE"])
        self._doc_filenames[doc_id] = chunks[0].source_file
        self._doc_chunk_counts[doc_id] = len(chunks)

    def remove_document(self, doc_id: str) -> bool:
        self.index.delete(filter={"doc_id": doc_id}, namespace=CONFIG["PINECONE_NAMESPACE"])
        self._doc_filenames.pop(doc_id, None); self._doc_chunk_counts.pop(doc_id, None)
        return True

    def search(self, query_embedding: np.ndarray, top_k: int = 20) -> list[SearchResult]:
        result = self.index.query(vector=query_embedding.flatten().tolist(),
                                  top_k=top_k, include_metadata=True,
                                  namespace=CONFIG["PINECONE_NAMESPACE"])
        results = []
        for i, match in enumerate(result.matches):
            meta = match.metadata or {}
            chunk = DocumentChunk(text=meta.get("text", ""), source_file=meta.get("filename", ""),
                                  page_number=int(meta.get("page_number", 0)),
                                  chunk_index=int(meta.get("chunk_index", 0)),
                                  doc_id=meta.get("doc_id", ""))
            results.append(SearchResult(chunk=chunk, score=match.score, global_index=i))
        return results

    def get_document_info(self) -> dict[str, dict]:
        return {doc_id: {"filename": name, "file_hash": doc_id,
                         "chunk_count": self._doc_chunk_counts.get(doc_id, 0)}
                for doc_id, name in self._doc_filenames.items()}

    def has_document(self, doc_id: str) -> bool: return doc_id in self._doc_filenames
    def get_doc_id_by_filename(self, filename: str) -> Optional[str]:
        for doc_id, name in self._doc_filenames.items():
            if name == filename: return doc_id
        return None
    def save(self): pass
    def load(self) -> bool: return True

_PINECONE_AVAILABLE = HAS_PINECONE and bool(CONFIG["PINECONE_API_KEY"])
print(f"PineconeVectorStore: {'available' if _PINECONE_AVAILABLE else 'not configured'}")


## 9. Retrieval Pipeline (RRF, MMR, Reranker)


In [ ]:
_reranker_model = None

def get_reranker():
    global _reranker_model
    if CONFIG["RERANK_PROVIDER"] == "HUGGINGFACE_API": return None
    if _reranker_model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _reranker_model = CrossEncoder(CONFIG["HF_RERANKER_MODEL_ID"], device=device)
    return _reranker_model

def reciprocal_rank_fusion(ranked_lists: list[list[tuple]], k: int = None) -> list[tuple]:
    k = k or CONFIG["RRF_K"]
    fused_scores, fused_data = {}, {}
    for rl in ranked_lists:
        for rank, (ident, chunk) in enumerate(rl):
            if ident not in fused_scores:
                fused_scores[ident] = 0.0; fused_data[ident] = chunk
            fused_scores[ident] += 1.0 / (k + rank + 1)
    sorted_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [(ident, score, fused_data[ident]) for ident, score in sorted_results]

def _rerank_candidates(query: str, candidates: list[tuple]) -> list[float]:
    if not candidates: return []
    try:
        if CONFIG["RERANK_PROVIDER"] == "HUGGINGFACE_API":
            client = _get_hf_client()
            return client.rerank(query, [c.text for _, _, c in candidates],
                                 model=CONFIG["HF_RERANKER_MODEL_ID"])
        reranker = get_reranker()
        pairs = [(query, chunk.text) for _, _, chunk in candidates]
        return list(reranker.predict(pairs))
    except Exception:
        return [score for _, score, _ in candidates]

def mmr_select(query_emb: np.ndarray, doc_embs: np.ndarray, chunks: list,
               top_k: int, lambda_mult: float = 0.5) -> list[tuple[int, float]]:
    if len(chunks) <= top_k:
        sims = doc_embs @ query_emb
        return [(i, float(sims[i])) for i in range(len(chunks))]
    q_sims = doc_embs @ query_emb; d_sims = doc_embs @ doc_embs.T
    selected, remaining = [], list(range(len(chunks)))
    best = remaining[int(np.argmax([q_sims[i] for i in remaining]))]
    selected.append(best); remaining.remove(best)
    while len(selected) < top_k and remaining:
        best_mmr, best_c = -float('inf'), None
        for c in remaining:
            rel = q_sims[c]; div = max(d_sims[c, s] for s in selected)
            mmr = lambda_mult * rel - (1 - lambda_mult) * div
            if mmr > best_mmr: best_mmr, best_c = mmr, c
        if best_c is not None: selected.append(best_c); remaining.remove(best_c)
    return [(i, float(q_sims[i])) for i in selected]

def apply_mmr(results: list[SearchResult], query_emb: np.ndarray,
              top_k: int, lam: float = 0.5) -> list[SearchResult]:
    if len(results) <= top_k: return results
    texts = [r.chunk.text for r in results]
    docs_emb = embed_texts(texts, normalize=True)
    indices = mmr_select(query_emb.flatten(), docs_emb, texts, top_k, lam)
    mmr_res = []
    for rank, (orig_idx, score) in enumerate(indices):
        r = results[orig_idx]; r.score = score; r.global_index = rank; mmr_res.append(r)
    return mmr_res

print("Retrieval primitives ready.")


## 10. Hybrid Search


In [ ]:
def hybrid_search(query: str, vector_store, bm25_store=None,
                  top_k: int = None, rerank_top_k: int = None,
                  use_mmr: bool = None, mmr_lambda: float = None) -> list[SearchResult]:
    top_k = top_k or CONFIG["RETRIEVAL_TOP_K"]
    rerank_top_k = rerank_top_k or CONFIG["RERANK_TOP_K"]
    use_mmr = CONFIG["MMR_ENABLED"] if use_mmr is None else use_mmr
    mmr_lambda = mmr_lambda if mmr_lambda is not None else CONFIG["MMR_LAMBDA"]

    query_emb = encode_query(query)
    dense = vector_store.search(query_emb, top_k=top_k)
    dense_ranked = [(f"{r.chunk.doc_id}_{r.chunk.chunk_index}", r.chunk) for r in dense]

    ranked = [dense_ranked]
    if CONFIG["SPARSE_RETRIEVAL"] and bm25_store is not None:
        sparse = bm25_store.search(query, top_k=top_k)
        ranked.append([(f"{c.doc_id}_{c.chunk_index}", c) for _, _, c in sparse])

    fused = reciprocal_rank_fusion(ranked)
    if not fused: return []

    if use_mmr and len(fused) > rerank_top_k:
        fused_results = [SearchResult(chunk=c, score=s, global_index=i)
                         for i, (_, s, c) in enumerate(fused)]
        fused_results = apply_mmr(fused_results, query_emb,
                                  min(top_k, len(fused)), mmr_lambda)
        fused = [(f"{r.chunk.doc_id}_{r.chunk.chunk_index}", r.score, r.chunk)
                 for r in fused_results]

    candidates = fused[:top_k]
    rerank_scores = _rerank_candidates(query, candidates)
    reranked = [SearchResult(chunk=c, score=float(rerank_scores[i]), global_index=i)
                for i, (_, _, c) in enumerate(candidates)]
    reranked.sort(key=lambda r: r.score, reverse=True)
    return reranked[:rerank_top_k]

def multi_query_hybrid_search(queries: list[str], vector_store, bm25_store=None,
                               top_k: int = None, rerank_top_k: int = None) -> list[SearchResult]:
    top_k = top_k or CONFIG["RETRIEVAL_TOP_K"]
    rerank_top_k = rerank_top_k or CONFIG["RERANK_TOP_K"]
    all_results = []
    for q in queries:
        all_results.append(hybrid_search(q, vector_store, bm25_store, top_k=top_k,
                                         rerank_top_k=rerank_top_k, use_mmr=False))
    seen, fused = set(), []
    for results in all_results:
        for r in results:
            cid = f"{r.chunk.doc_id}_{r.chunk.chunk_index}"
            if cid not in seen:
                seen.add(cid); fused.append(r)
                if len(fused) >= rerank_top_k * 2: break
        if len(fused) >= rerank_top_k * 2: break
    if len(fused) > rerank_top_k:
        candidates = [(f"{r.chunk.doc_id}_{r.chunk.chunk_index}", r.score, r.chunk)
                      for r in fused[:top_k]]
        scores = _rerank_candidates(queries[0], candidates)
        for i, r in enumerate(fused[:top_k]): r.score = float(scores[i])
        fused.sort(key=lambda r: r.score, reverse=True)
        fused = fused[:rerank_top_k]
    for i, r in enumerate(fused): r.global_index = i
    return fused

print("Hybrid search ready.")


## 11. Query Enhancement
### 11a. Query Rewriting
### 11b. Multi-Query Expansion


In [ ]:
REWRITE_SYSTEM = (
    "You are a query optimizer for document search. "
    "Your ONLY job is to fix obvious typos. "
    "NEVER add words, NEVER expand acronyms, NEVER change meaning. "
    "Return ONLY the corrected query or the original if no typos."
)
REWRITE_USER = """Fix ONLY obvious typos in this query. Rules:
1. Fix misspelled words ONLY
2. Do NOT add ANY words
3. Do NOT expand acronyms
4. If no typos, return EXACTLY the original query
5. Return ONLY the query, nothing else

Query: {query}

Result:"""

def rewrite_query(query: str) -> str:
    if not CONFIG["QUERY_REWRITE_ENABLED"]: return query
    if len(query.split()) <= 4: return query
    if query.endswith('?') and len(query.split()) <= 8: return query
    try:
        msgs = [{"role": "system", "content": REWRITE_SYSTEM},
                {"role": "user", "content": REWRITE_USER.format(query=query)}]
        rewritten = generate(msgs, max_tokens=30, temperature=0.0).strip().strip('"').strip("'")
        if not rewritten or len(rewritten) > 150: return query
        orig_tokens = query.split()
        new_tokens = rewritten.split()
        if len(new_tokens) > len(orig_tokens): return query
        orig_digits = re.findall(r"\d+", query)
        new_digits = re.findall(r"\d+", rewritten)
        if orig_digits and new_digits != orig_digits: return query
        overlap = len(set(query.lower().split()) & set(rewritten.lower().split()))
        if overlap < len(orig_tokens) * 0.7: return query
        return rewritten if rewritten.lower() != query.lower() else query
    except Exception: return query

MULTI_QUERY_SYSTEM = (
    "You are a query expansion specialist. Generate 3 alternate versions "
    "of the user's question that preserve the original intent but use "
    "different wording, synonyms, or perspectives. "
    "Return ONLY the 3 queries, one per line, with no numbering."
)
MULTI_QUERY_USER = """Generate 3 alternate search queries for this question.
Rules:
1. Preserve the original meaning and intent
2. Use different wording, synonyms, or perspectives
3. Each query should be self-contained
4. Return ONLY the 3 queries, one per line

Original: {query}

Alternates:"""

def generate_alternate_queries(query: str, num_alternates: int = 3) -> list[str]:
    if len(query.split()) <= 2: return [query]
    try:
        msgs = [{"role": "system", "content": MULTI_QUERY_SYSTEM},
                {"role": "user", "content": MULTI_QUERY_USER.format(query=query)}]
        resp = generate(msgs, max_tokens=150, temperature=0.3)
        alts = []
        for line in resp.strip().split('\n'):
            line = line.strip().strip('"').strip("'").strip('-').strip()
            cleaned = line.lstrip('0123456789.-) ')
            if cleaned and len(cleaned) > 5 and cleaned.lower() != query.lower():
                alts.append(cleaned)
        seen, unique = set(), []
        for a in alts:
            if a.lower() not in seen and len(unique) < num_alternates:
                seen.add(a.lower()); unique.append(a)
        return [query] + unique
    except Exception: return [query]

print("Query enhancement ready.")


## 12. Context Compression


In [ ]:
def compress_context(context_chunks: list[dict[str, str]], query: str,
                    compression_ratio: float = None, min_chunks: int = None) -> list[dict[str, str]]:
    compression_ratio = compression_ratio if compression_ratio is not None else CONFIG["COMPRESSION_RATIO"]
    min_chunks = min_chunks if min_chunks is not None else CONFIG["MIN_CONTEXT_CHUNKS"]
    if len(context_chunks) <= min_chunks: return context_chunks
    try:
        query_emb = embed_texts([query], normalize=True)
        texts = [c["text"] for c in context_chunks]
        chunk_embs = embed_texts(texts, normalize=True)
        sims = (chunk_embs @ query_emb.T).flatten()
        scored = [{**c, "_score": float(sims[i]), "_idx": i} for i, c in enumerate(context_chunks)]
        scored.sort(key=lambda x: x["_score"], reverse=True)
        keep = max(min_chunks, int(len(context_chunks) * compression_ratio))
        kept = scored[:keep]
        kept.sort(key=lambda x: x["_idx"])
        for k in kept: k.pop("_score", None); k.pop("_idx", None)
        return kept
    except Exception: return context_chunks

def extract_key_sentences(text: str, query: str, max_sentences: int = 5) -> str:
    sentences, current = [], ""
    for ch in text:
        current += ch
        if ch in '.!?' and len(current.strip()) > 10:
            sentences.append(current.strip()); current = ""
    if current.strip(): sentences.append(current.strip())
    if len(sentences) <= max_sentences: return text
    try:
        q_emb = embed_texts([query], normalize=True)
        s_embs = embed_texts(sentences, normalize=True)
        scores = (s_embs @ q_emb.T).flatten()
        top = np.argsort(scores)[-max_sentences:][::-1]
        return " ".join(sentences[i] for i in sorted(top))
    except Exception: return text

print("Context compression ready.")


## 13. Intent Classification


In [ ]:
_OOS_PATTERNS = [
    (re.compile(r'\b(calculate|compute|solve|evaluate)\b.*\b(equation|integral|derivative|matrix|polynomial)\b', re.I), "math"),
    (re.compile(r'\bwhat\s+is\s+\d+\s*[\+\-\*/\^]\s*\d+', re.I), "math"),
    (re.compile(r'\b(my|your|his|her)\s+(name|age|birthday|address|phone|email)\b', re.I), "personal"),
    (re.compile(r'\bwho\s+am\s+i\b', re.I), "personal"),
    (re.compile(r'\b(capital|president|prime minister|population|gdp)\s+of\s+(france|china|india|usa|japan|germany|brazil)', re.I), "general"),
    (re.compile(r'\b(who\s+won|what\s+happened)\s+(in\s+)?(the\s+)?(world\s+cup|olympics|super\s+bowl|election)', re.I), "general"),
    (re.compile(r'\b(explain|what\s+is|define)\s+(quantum\s+physics|theory\s+of\s+relativity|general\s+relativity|special\s+relativity|black\s+hole)', re.I), "general"),
]

INTENT_STRATEGIES = {
    "extraction":     {"top_k": 15, "rerank_top_k": 8,  "use_multi_query": True,  "compression_ratio": 0.6, "structured_output": True,  "requires_verification": True},
    "counting":       {"top_k": 12, "rerank_top_k": 6,  "use_multi_query": True,  "compression_ratio": 0.5, "structured_output": True,  "requires_verification": True},
    "comparison":     {"top_k": 15, "rerank_top_k": 8,  "use_multi_query": True,  "compression_ratio": 0.7, "structured_output": False, "requires_verification": True},
    "summarization":  {"top_k": 10, "rerank_top_k": 5,  "use_multi_query": False, "compression_ratio": 0.8, "structured_output": False, "requires_verification": False},
    "verification":   {"top_k": 8,  "rerank_top_k": 4,  "use_multi_query": False, "compression_ratio": 0.9, "structured_output": True,  "requires_verification": True},
    "table_parsing":  {"top_k": 10, "rerank_top_k": 5,  "use_multi_query": False, "compression_ratio": 0.7, "structured_output": True,  "requires_verification": True},
    "qa":             {"top_k": 12, "rerank_top_k": 8,  "use_multi_query": False, "compression_ratio": 1.0, "structured_output": False, "requires_verification": False},
    "out_of_scope":   {"top_k": 0,  "rerank_top_k": 0,  "use_multi_query": False, "compression_ratio": 0.0, "structured_output": False, "requires_verification": False},
}

VALID_INTENTS = set(INTENT_STRATEGIES.keys())

def classify_intent(query: str) -> dict:
    q_lower = query.lower()
    for pat, stype in _OOS_PATTERNS:
        if pat.search(query):
            return {"intent": "out_of_scope", "strategy": INTENT_STRATEGIES["out_of_scope"],
                    "params": {"scope_type": stype}, "is_out_of_scope": True}
    if any(kw in q_lower for kw in ["extract", "list all", "get all", "pull", "give me list"]):
        intent = "extraction"
    elif any(kw in q_lower for kw in ["how many", "count", "total number", "number of"]):
        intent = "counting"
    elif any(kw in q_lower for kw in ["compare", "difference between", " vs ", "versus"]):
        intent = "comparison"
    elif any(kw in q_lower for kw in ["summarize", "overview", "what is this document", "what is this about"]):
        intent = "summarization"
    elif any(kw in q_lower for kw in ["is there", "does the document", "verify", "check if", "is mentioned"]):
        intent = "verification"
    elif any(kw in q_lower for kw in ["table", "tabular", "in table format"]):
        intent = "table_parsing"
    else: intent = "qa"

    if len(query.split()) > 8:
        try:
            msgs = [{"role": "system", "content": "You are an intent classifier. Return ONLY the category name: extraction|counting|comparison|summarization|verification|table_parsing|qa|out_of_scope"},
                    {"role": "user", "content": f"Classify: {query}"}]
            llm_intent = generate(msgs, max_tokens=20, temperature=0.0).strip().lower()
            if llm_intent in VALID_INTENTS: intent = llm_intent
        except Exception: pass

    return {"intent": intent, "strategy": INTENT_STRATEGIES[intent],
            "params": _extract_query_params(q_lower, intent), "is_out_of_scope": False}

def _extract_query_params(query: str, intent: str) -> dict:
    params = {}
    m = re.search(r'(?:maximum|max|up to|at most)\s+(\d+)', query)
    if m: params["max_limit"] = int(m.group(1))
    if "all" in query or "every" in query: params["extract_all"] = True
    if intent in ("extraction", "counting"):
        for kw in ["extract", "list", "get", "pull", "count", "how many"]:
            if kw in query:
                after = query[query.find(kw) + len(kw):].strip()
                for f in ["the", "all", "from", "document", "pdf", "file"]: after = after.replace(f, "").strip()
                if after: params["target_entity"] = after; break
    return params

print("Intent classifier ready.")


## 14. Task Prompts


In [ ]:
EXTRACTION_SYSTEM = "You are a precise data extraction assistant. Extract ONLY from context. Return JSON with items array."
COUNTING_SYSTEM = "You are a precise counting assistant. Count ONLY from context. Return JSON with count."
COMPARISON_SYSTEM = "You are a precise comparison assistant. Compare ONLY using context."
SUMMARIZATION_SYSTEM = "You are an expert document analyst. Provide comprehensive summaries using ONLY context."
VERIFICATION_SYSTEM = "You are a precise verification assistant. Verify claims against context. Return JSON."
TABLE_SYSTEM = "You are a precise table extraction assistant. Extract tabular data as JSON."
QA_SYSTEM = CONFIG["SYSTEM_PROMPT"]

def build_task_prompt(intent: str, context: str, query: str, params: dict = None) -> list[dict[str, str]]:
    params = params or {}
    if intent == "extraction":
        target = params.get("target_entity", "requested information")
        limit = ""
        if params.get("max_limit"): limit = f"IMPORTANT: Extract MAXIMUM {params['max_limit']} items."
        elif params.get("extract_all"): limit = "Extract ALL instances found."
        user = f"Extract {target} from context. {limit}\n\nContext:\n{context}\n\nReturn ONLY a JSON object."
        sys = EXTRACTION_SYSTEM
    elif intent == "counting":
        entity = params.get("target_entity", "items")
        user = f"Count {entity} in context.\n\nContext:\n{context}\n\nReturn ONLY a JSON object."
        sys = COUNTING_SYSTEM
    elif intent == "comparison":
        user = f"Compare entities.\n\nContext:\n{context}\n\nProvide structured comparison with citations."
        sys = COMPARISON_SYSTEM
    elif intent == "summarization":
        user = f"Provide comprehensive summary.\n\nContext:\n{context}"
        sys = SUMMARIZATION_SYSTEM
    elif intent == "verification":
        user = f"Verify claim: {query}\n\nContext:\n{context}\n\nReturn ONLY a JSON object."
        sys = VERIFICATION_SYSTEM
    elif intent == "table_parsing":
        user = f"Extract tabular data.\n\nContext:\n{context}\n\nReturn ONLY JSON with headers and rows."
        sys = TABLE_SYSTEM
    else:
        user = f"Context:\n{context}\n\n---\n\nQuestion: {query}\n\nAnswer based ONLY on the context above."
        sys = QA_SYSTEM
    return [{"role": "system", "content": sys}, {"role": "user", "content": user}]

print("Task prompts ready.")


## 15. LLM & HuggingFace API Client


In [ ]:
_llm_model = None; _llm_tokenizer = None

def get_llm():
    global _llm_model, _llm_tokenizer
    if CONFIG["LLM_PROVIDER"] != "HUGGINGFACE": return None, None
    if _llm_model is None:
        model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
        _llm_tokenizer = AutoTokenizer.from_pretrained(model_id)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _llm_model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else "cpu")
        _llm_model.eval()
        if _llm_tokenizer.pad_token is None: _llm_tokenizer.pad_token = _llm_tokenizer.eos_token
    return _llm_model, _llm_tokenizer

def _format_hf_prompt(messages: list[dict]) -> str:
    prompt = ""
    for m in messages:
        if m["role"] == "system": prompt += f"<|system|>\n{m['content']}</s>\n"
        elif m["role"] == "user": prompt += f"<|user|>\n{m['content']}</s>\n"
        elif m["role"] == "assistant": prompt += f"<|assistant|>\n{m['content']}</s>\n"
    prompt += "<|assistant|>\n"
    return prompt

def _generate_hf_local(messages: list[dict], max_tokens: int = None, temperature: float = None):
    max_tokens = max_tokens or CONFIG["LLM_MAX_TOKENS"]
    temperature = temperature if temperature is not None else CONFIG["LLM_TEMPERATURE"]
    model, tokenizer = get_llm()
    prompt = _format_hf_prompt(messages)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**inputs, streamer=streamer, max_new_tokens=max_tokens,
                  do_sample=temperature > 0, temperature=max(temperature, 0.01),
                  top_p=0.9, repetition_penalty=1.1)
    thread = Thread(target=model.generate, kwargs=kwargs)
    thread.start()
    for t in streamer:
        if t: yield t
    thread.join()

def _generate_ollama(messages: list[dict], max_tokens: int = None, temperature: float = None):
    max_tokens = max_tokens or CONFIG["LLM_MAX_TOKENS"]
    temperature = temperature if temperature is not None else CONFIG["LLM_TEMPERATURE"]
    payload = {"model": CONFIG["OLLAMA_MODEL"], "messages": messages,
               "stream": True, "options": {"temperature": temperature, "num_predict": max_tokens}}
    with httpx.stream("POST", f"{CONFIG['OLLAMA_BASE_URL']}/api/chat", json=payload, timeout=120.0) as resp:
        if resp.status_code != 200: raise RuntimeError(f"Ollama error {resp.status_code}")
        for line in resp.iter_lines():
            if not line: continue
            try: data = json.loads(line)
            except json.JSONDecodeError: continue
            if "message" in data and "content" in data["message"]: yield data["message"]["content"]
            if data.get("done"): break

def _generate_hf_api(messages: list[dict], max_tokens: int = None, temperature: float = None):
    max_tokens = max_tokens or CONFIG["LLM_MAX_TOKENS"]
    temperature = temperature if temperature is not None else CONFIG["LLM_TEMPERATURE"]
    client = _get_hf_client()
    stream = client.chat.completions.create(
        model=CONFIG["HF_CHAT_MODEL_ID"], messages=messages,
        max_tokens=max_tokens, temperature=temperature, stream=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

def generate_stream(messages: list[dict], max_tokens: int = None, temperature: float = None):
    if CONFIG["LLM_PROVIDER"] == "OLLAMA": yield from _generate_ollama(messages, max_tokens, temperature)
    elif CONFIG["LLM_PROVIDER"] == "HUGGINGFACE_API": yield from _generate_hf_api(messages, max_tokens, temperature)
    else: yield from _generate_hf_local(messages, max_tokens, temperature)

def generate(messages: list[dict], max_tokens: int = None, temperature: float = None) -> str:
    return "".join(generate_stream(messages, max_tokens, temperature))

print(f"LLM provider: {CONFIG['LLM_PROVIDER']}")


## 16. Verification


In [ ]:
def _split_into_claims(text: str) -> list[str]:
    sentences, current = [], ""
    for ch in text:
        current += ch
        if ch in ".!?" and len(current.strip()) > 10: sentences.append(current.strip()); current = ""
    if current.strip() and len(current.strip()) > 10: sentences.append(current.strip())
    return sentences or [text.strip()]

def verify_extraction(items: list[dict], context_texts: list[str], min_conf: float = None) -> tuple[bool, float, list[str]]:
    min_conf = min_conf or CONFIG["FAITHFULNESS_THRESHOLD"]
    if not items: return True, 1.0, []
    unsupported, scores = [], []
    for item in items:
        item_text = str(item.get("name", "")) + " " + str(item.get("details", ""))
        if not item_text.strip(): continue
        supported, best = False, 0.0
        for ctx in context_texts:
            overlap = len(set(item_text.lower().split()) & set(ctx.lower().split()))
            if overlap >= 2:
                try:
                    ie = embed_texts([item_text], normalize=True)
                    ce = embed_texts([ctx], normalize=True)
                    sim = float(ie @ ce.T)
                    if sim > best: best = sim
                    if sim >= min_conf: supported = True; break
                except Exception:
                    if overlap >= 3: supported = True; best = overlap / max(len(item_text.split()), 1); break
        scores.append(best)
        if not supported: unsupported.append(item_text)
    if not scores: return True, 1.0, []
    return len(unsupported) == 0, sum(scores) / len(scores), unsupported

def verify_answer(answer: str, context_texts: list[str], min_conf: float = None) -> tuple[bool, float]:
    min_conf = min_conf or CONFIG["FAITHFULNESS_THRESHOLD"]
    if not answer.strip() or not context_texts: return False, 0.0
    if any(kw in answer.lower() for kw in ["not found in document", "no information", "i don't have",
                                           "the uploaded documents do not contain"]):
        return True, 1.0
    if CONFIG["VERIFICATION_PROVIDER"] == "HUGGINGFACE_API":
        client = _get_hf_client()
        resp = client.chat.completions.create(
            model=CONFIG["HF_VERIFIER_MODEL_ID"],
            messages=[{"role": "system", "content": "Determine if the answer comes from the context. Answer YES or NO."},
                      {"role": "user", "content": f"Context: {textwrap.shorten(' '.join(context_texts), 3000)}\n\nAnswer: {answer}"}],
            max_tokens=5, temperature=0.0)
        result = resp.choices[0].message.content.strip().upper()
        return "YES" in result, 0.85 if "YES" in result else 0.15
    claims = [c for c in _split_into_claims(answer)
              if not c.lower().startswith(("not found", "i don't", "no information", "the document does not"))]
    if not claims: return True, 1.0
    scores = []
    for claim in claims:
        best = 0.0
        for ctx in context_texts:
            cw = set(claim.lower().split())
            ctxw = set(ctx.lower().split())
            overlap = len(cw & ctxw) / max(len(cw), 1)
            if overlap >= 0.3:
                try:
                    ie = embed_texts([claim], normalize=True)
                    ce = embed_texts([ctx], normalize=True)
                    sim = float(ie @ ce.T)
                    if sim > best: best = sim
                except Exception: best = max(best, overlap * 0.8)
        if best < min_conf: best = max(best, min(overlap * 0.7, 0.5))
        scores.append(best)
    if not scores: return False, 0.0
    scores.sort()
    final = 0.6 * (sum(scores)/len(scores)) + 0.4 * scores[len(scores)//2]
    return final >= (min_conf * 0.8), final

def verify_count(claimed: int, entity: str, context_texts: list[str]) -> tuple[bool, str]:
    entity_lower = entity.lower()
    found = any(entity_lower in ctx.lower() for ctx in context_texts)
    if claimed == 0:
        return (False, f"Entity '{entity}' appears in context but count is 0") if found else (True, "Not found, count 0")
    return (True, "Plausible") if found else (False, f"Entity '{entity}' not found")

print("Verification ready.")


## 17. NLI Faithfulness Verifier


In [ ]:
_nli_model = None; _nli_tokenizer = None

def get_nli_model():
    global _nli_model, _nli_tokenizer
    if _nli_model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _nli_tokenizer = AutoTokenizer.from_pretrained(CONFIG["NLI_MODEL"])
        _nli_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["NLI_MODEL"])
        _nli_model.to(device); _nli_model.eval()
    return _nli_model, _nli_tokenizer

def _check_entailment(claim: str, context: str) -> float:
    model, tokenizer = get_nli_model()
    device = next(model.parameters()).device
    max_ctx = tokenizer.model_max_length - len(tokenizer(claim)["input_ids"]) - 3
    if max_ctx > 0:
        ctx_tokens = tokenizer(context, truncation=True, max_length=max_ctx, return_tensors="pt")
        context = tokenizer.decode(ctx_tokens["input_ids"][0], skip_special_tokens=True)
    inputs = tokenizer(context, claim, return_tensors="pt", truncation=True,
                       max_length=tokenizer.model_max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
    return probs[0].item()

def verify_faithfulness(answer: str, context_texts: list[str], threshold: float = None) -> tuple[bool, float]:
    threshold = threshold or CONFIG["NLI_ENTAILMENT_THRESHOLD"]
    if not answer.strip() or not context_texts: return False, 0.0
    if any(kw in answer.lower() for kw in ["not found in document", "no information", "i don't have",
                                           "the uploaded documents do not contain"]):
        return True, 1.0
    if CONFIG["VERIFICATION_PROVIDER"] == "HUGGINGFACE_API":
        return verify_answer(answer, context_texts)
    claims = _split_into_claims(answer)
    full_ctx = "\n\n".join(context_texts)
    scores = []
    for claim in claims:
        try: scores.append(_check_entailment(claim, full_ctx))
        except Exception: scores.append(0.0)
    if not scores: return False, 0.0
    avg = sum(scores) / len(scores)
    return avg >= threshold, avg

print("NLI verifier ready.")


## 18. ReasoningAgent


In [ ]:
_FALSE_NOT_FOUND_PATTERNS = [
    re.compile(r'^The document does not provide', re.I | re.M),
    re.compile(r'^The uploaded documents do not contain', re.I | re.M),
    re.compile(r'^Not found in document', re.I | re.M),
    re.compile(r"^I don.t have enough information", re.I | re.M),
]

class ReasoningAgent:
    def __init__(self):
        self.intent_cache: dict[str, dict] = {}

    def process(self, query: str, context_chunks: list[dict[str, str]],
                history: list[dict] = None) -> dict:
        intent_info = classify_intent(query)
        intent = intent_info["intent"]; strategy = intent_info["strategy"]; params = intent_info["params"]

        if intent == "out_of_scope":
            msg = f"I can only answer about uploaded documents. This appears to be a {params.get('scope_type', 'unknown')} query."
            return {"answer": msg, "verification": {"is_grounded": True, "confidence": 1.0, "out_of_scope": True},
                    "intent": intent, "tokens": list(msg), "citations": [], "structured": None}

        context_text = self._format_context(context_chunks)
        context_texts = [c["text"] for c in context_chunks]
        messages = build_task_prompt(intent, context_text, query, params)

        max_tokens = CONFIG["LLM_MAX_TOKENS_COMPREHENSIVE"] if intent in ("qa", "summarization") else CONFIG["LLM_MAX_TOKENS"]
        full_response = "".join(generate_stream(messages, max_tokens=max_tokens))
        full_response = self._post_process_response(full_response)

        verification = self._verify_response(intent, full_response, context_texts, params)
        structured = self._parse_structured_output(intent, full_response, context_chunks) if strategy.get("structured_output") else None
        citations = self._extract_citations(context_chunks)

        return {"answer": full_response, "verification": verification,
                "structured": structured, "citations": citations,
                "intent": intent, "tokens": list(full_response)}

    def _format_context(self, chunks: list[dict[str, str]]) -> str:
        parts = []
        for i, c in enumerate(chunks, 1):
            parts.append(f"[{i}] Source: {c.get('source_file', 'unknown')}, Page: {c.get('page_number', '?')}\n{c.get('text', '')}")
        return "\n\n---\n\n".join(parts)

    def _post_process_response(self, response: str) -> str:
        lines = [l for l in response.split("\n") if not any(p.match(l.strip()) for p in _FALSE_NOT_FOUND_PATTERNS)]
        result = "\n".join(lines).strip()
        return result or "The uploaded documents do not contain this information."

    def _verify_response(self, intent: str, response: str, context_texts: list[str], params: dict) -> dict:
        result = {"is_grounded": True, "confidence": 0.0, "issues": []}
        if any(kw in response.lower() for kw in ["not found in document", "no information", "i don't have",
                                                  "the uploaded documents do not contain"]):
            result["confidence"] = 1.0; return result
        if intent == "extraction":
            items = self._extract_items_from_response(response)
            if items:
                valid, conf, unsup = verify_extraction(items, context_texts)
                result["is_grounded"] = valid; result["confidence"] = conf
                if unsup: result["issues"].append(f"{len(unsup)} items may not be fully supported")
        elif intent == "counting":
            count = self._extract_count_from_response(response)
            entity = params.get("target_entity", "items")
            accurate, msg = verify_count(count, entity, context_texts)
            result["is_grounded"] = accurate; result["confidence"] = 1.0 if accurate else 0.3
            if not accurate: result["issues"].append(msg)
        else:
            faithful, conf = verify_answer(response, context_texts)
            result["is_grounded"] = faithful; result["confidence"] = conf
            if not faithful: result["issues"].append("Answer may not be fully supported")
        return result

    def _parse_structured_output(self, intent: str, response: str, chunks: list[dict]) -> Optional[dict]:
        try:
            start = response.find('{'); end = response.rfind('}')
            if start != -1 and end > start:
                data = json.loads(response[start:end+1])
                return validate_structured_output(data, intent)
        except Exception: pass
        return None

    def _extract_items_from_response(self, response: str) -> list[dict]:
        try:
            start, end = response.find('{'), response.rfind('}')
            if start != -1 and end > start:
                return json.loads(response[start:end+1]).get("items", [])
        except Exception: pass
        return []

    def _extract_count_from_response(self, response: str) -> int:
        try:
            start, end = response.find('{'), response.rfind('}')
            if start != -1 and end > start:
                return json.loads(response[start:end+1]).get("count", 0)
        except Exception: pass
        m = re.search(r'count[:\s]*(\d+)', response.lower())
        return int(m.group(1)) if m else 0

    def _extract_citations(self, chunks: list[dict]) -> list[dict]:
        citations = []
        for i, c in enumerate(chunks):
            text = c.get("text", "")
            snippet = text[:200] + ("..." if len(text) > 200 else "")
            citations.append({"index": i+1, "source_file": c.get("source_file", "unknown"),
                             "page_number": c.get("page_number", "?"), "text_snippet": snippet})
        return citations

print("ReasoningAgent ready.")


## 19. Query Cache


In [ ]:
class QueryCache:
    def __init__(self, ttl: int = 3600, max_size: int = 100):
        self.ttl = ttl; self.max_size = max_size
        (INDEX_DIR / "query_cache").mkdir(parents=True, exist_ok=True)

    def _key(self, question: str, doc_count: int) -> str:
        return hashlib.md5(f"{question.lower().strip()}|docs:{doc_count}".encode()).hexdigest()

    def get(self, question: str, doc_count: int) -> Optional[dict]:
        path = INDEX_DIR / "query_cache" / f"{self._key(question, doc_count)}.json"
        if not path.exists(): return None
        try:
            data = json.loads(path.read_text())
            if time.time() - data.get("timestamp", 0) > self.ttl: path.unlink(); return None
            return data.get("response")
        except Exception: return None

    def put(self, question: str, doc_count: int, response: dict):
        path = INDEX_DIR / "query_cache" / f"{self._key(question, doc_count)}.json"
        path.write_text(json.dumps({"question": question, "timestamp": time.time(),
                                    "doc_count": doc_count, "response": response}))
        self._enforce()

    def _enforce(self):
        files = sorted((INDEX_DIR / "query_cache").glob("*.json"), key=lambda f: f.stat().st_mtime)
        for f in files[:-self.max_size]: f.unlink()

    def clear(self):
        for f in (INDEX_DIR / "query_cache").glob("*.json"): f.unlink()

_cache = None
def get_cache():
    global _cache
    if _cache is None: _cache = QueryCache()
    return _cache

print("QueryCache ready.")


## 20. RAGPipeline (The Core Orchestrator)


In [ ]:
class RAGPipeline:
    def __init__(self):
        self.vector_store = create_vector_store()
        self.bm25_store = BM25Store() if CONFIG["SPARSE_RETRIEVAL"] else None
        self.reasoning_agent = ReasoningAgent()
        self._models_loaded = False

    def load_indices(self):
        self.vector_store.load()
        if self.bm25_store: self.bm25_store.load()

    def save_indices(self):
        self.vector_store.save()
        if self.bm25_store: self.bm25_store.save()

    def load_models(self):
        remote = all(CONFIG.get(p) == "HUGGINGFACE_API"
                     for p in ["LLM_PROVIDER", "EMBEDDING_PROVIDER", "RERANK_PROVIDER", "VERIFICATION_PROVIDER"])
        if remote: self._models_loaded = True; return
        get_embed_model(); get_llm(); self._models_loaded = True

    @property
    def models_loaded(self): return self._models_loaded

    def _compute_file_hash(self, file_path: Path) -> str:
        sha = hashlib.sha256()
        with open(file_path, "rb") as f:
            for block in iter(lambda: f.read(8192), b""): sha.update(block)
        return sha.hexdigest()

    def ingest(self, file_path: Path, filename: str) -> dict:
        file_hash = self._compute_file_hash(file_path)
        if self.vector_store.has_document(file_hash):
            count = len(self.vector_store.doc_index.get(file_hash, []))
            return {"filename": filename, "chunk_count": count, "status": "unchanged", "file_hash": file_hash}
        existing = self.vector_store.get_doc_id_by_filename(filename)
        status = "indexed"
        if existing and existing != file_hash:
            self.vector_store.remove_document(existing)
            if self.bm25_store: self.bm25_store.remove_document(existing)
            status = "updated"
        sections = parse_document(file_path)
        if not sections: return {"filename": filename, "chunk_count": 0, "status": "empty", "file_hash": file_hash}
        chunks = chunk_sections(sections, doc_id=file_hash)
        if not chunks: return {"filename": filename, "chunk_count": 0, "status": "empty", "file_hash": file_hash}
        chunk_embeddings = embed_texts([c.text for c in chunks])
        self.vector_store.add(chunk_embeddings, chunks, doc_id=file_hash)
        if self.bm25_store: self.bm25_store.add(chunks, doc_id=file_hash)
        self.save_indices()
        return {"filename": filename, "chunk_count": len(chunks), "status": status, "file_hash": file_hash}

    def delete_document(self, filename: str) -> bool:
        doc_id = self.vector_store.get_doc_id_by_filename(filename)
        if not doc_id: return False
        self.vector_store.remove_document(doc_id)
        if self.bm25_store: self.bm25_store.remove_document(doc_id)
        self.save_indices()
        fp = DATA_DIR / "uploads" / filename
        if fp.exists(): fp.unlink()
        return True

    def get_documents(self) -> list[dict]:
        return [{"filename": v["filename"], "file_hash": k, "chunk_count": v["chunk_count"]}
                for k, v in self.vector_store.get_document_info().items()]

    def query(self, question: str, history: list[dict] = None) -> dict:
        doc_count = len(self.vector_store.get_document_info())
        cached = get_cache().get(question, doc_count)
        if cached: return cached

        intent_info = classify_intent(question)
        intent = intent_info["intent"]; strategy = intent_info["strategy"]; params = intent_info["params"]

        if intent == "out_of_scope":
            msg = f"I can only answer about uploaded documents. This appears to be a {params.get('scope_type', 'unknown')} query."
            result = {"answer": msg, "intent": intent, "citations": [],
                      "verification": {"is_grounded": True, "confidence": 1.0, "out_of_scope": True}, "structured": None}
            get_cache().put(question, doc_count, result); return result

        rewritten = rewrite_query(question)

        if strategy["use_multi_query"]:
            queries = generate_alternate_queries(rewritten)
            search_results = multi_query_hybrid_search(queries, self.vector_store, self.bm25_store,
                                                        top_k=strategy["top_k"], rerank_top_k=strategy["rerank_top_k"])
        else:
            search_results = hybrid_search(rewritten, self.vector_store, self.bm25_store,
                                            top_k=strategy["top_k"], rerank_top_k=strategy["rerank_top_k"])

        if not search_results:
            result = {"answer": "No relevant documents found.", "intent": intent, "citations": [],
                      "verification": {"is_grounded": True, "confidence": 1.0}, "structured": None}
            get_cache().put(question, doc_count, result); return result

        top_score = search_results[0].score
        if top_score < CONFIG["RETRIEVAL_NO_ANSWER_THRESHOLD"]:
            result = {"answer": "No relevant information found.", "intent": intent, "citations": [],
                      "verification": {"is_grounded": True, "confidence": 1.0,
                                       "issues": [f"Top score {top_score:.3f} below threshold"]}, "structured": None}
            get_cache().put(question, doc_count, result); return result

        context_chunks = [{"text": r.chunk.text, "source_file": r.chunk.source_file,
                          "page_number": str(r.chunk.page_number)} for r in search_results]
        compression_ratio = 1.0 if intent in ("qa", "summarization") else strategy.get("compression_ratio", 0.7)
        compressed = compress_context(context_chunks, rewritten, compression_ratio)

        agent_result = self.reasoning_agent.process(question, compressed, history)

        citations = []
        for i, r in enumerate(search_results):
            s = r.chunk.text[:200] + ("..." if len(r.chunk.text) > 200 else "")
            citations.append({"index": i+1, "source_file": r.chunk.source_file,
                             "page_number": r.chunk.page_number, "chunk_index": r.chunk.chunk_index,
                             "text_snippet": s, "relevance_score": round(r.score, 4)})

        verification = agent_result.get("verification")
        if not verification:
            faithful, score = verify_faithfulness(agent_result["answer"], [c["text"] for c in compressed])
            verification = {"is_grounded": faithful, "confidence": round(score, 3),
                            "issues": [] if faithful else ["Answer may not be fully supported"]}

        result = {"answer": agent_result["answer"], "intent": intent, "citations": citations,
                  "verification": verification, "structured": agent_result.get("structured"),
                  "rewritten_query": rewritten if rewritten != question else None}

        get_cache().put(question, doc_count, result)
        return result

print("RAGPipeline ready.")


## 21. RAG Evaluation (HF Cookbook Style)
### 21a. Evaluation Prompts


In [ ]:
QA_GENERATION_PROMPT = """Your task is to write a factoid question and an answer given a context.
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Your factoid question MUST NOT mention "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Now here is the context.

Context: {context}

Output:::"""

GROUNDEDNESS_PROMPT = """You are a fair evaluator. Rate the following QA pair on groundedness (1-5).
Groundedness: Is the answer fully supported by the context?

Context: {context}
Question: {question}
Answer: {answer}

Rate 1-5 where:
5 = Fully grounded in the context
4 = Mostly grounded
3 = Partially grounded
2 = Barely grounded
1 = Not grounded at all

Output format:
Groundedness score: <number>
Feedback: <your feedback>"""

RELEVANCE_PROMPT = """Rate the relevance of this question to the context (1-5).
Does the question make sense given the context?

Context: {context}
Question: {question}

Output format:
Relevance score: <number>"""

STANDALONE_PROMPT = """Rate whether this question is standalone and understandable without the context (1-5).

Question: {question}

Output format:
Standalone score: <number>"""

EVAL_PROMPT = """Rate the correctness of the generated answer compared to the reference answer (1-5).

Question: {question}
Reference Answer: {reference}
Generated Answer: {generated}

Score Rubric:
1 = Completely incorrect
2 = Mostly incorrect
3 = Somewhat correct
4 = Mostly correct
5 = Completely correct

Output format:
Score: <number>
Feedback: <your feedback>"""

print("Evaluation prompts defined.")


### 21b. Synthetic QA Generation


In [ ]:
def generate_qa_pairs(docs_processed: list[DocumentChunk], num_pairs: int = 20) -> list[dict]:
    outputs = []
    sampled = random.sample(docs_processed, min(num_pairs * 2, len(docs_processed)))
    for chunk in tqdm(sampled[:num_pairs], desc="Generating QA pairs"):
        try:
            msgs = [{"role": "user", "content": QA_GENERATION_PROMPT.format(context=chunk.text[:2000])}]
            resp = generate(msgs, max_tokens=200, temperature=0.3)
            if "Factoid question:" in resp and "Answer:" in resp:
                question = resp.split("Factoid question:")[-1].split("Answer:")[0].strip()
                answer = resp.split("Answer:")[-1].strip()
                if len(answer) > 5:
                    outputs.append({
                        "question": question, "answer": answer,
                        "context": chunk.text, "source_file": chunk.source_file
                    })
        except Exception as e:
            print(f"Error generating QA: {e}")
    return outputs

def filter_qa_pairs(qa_pairs: list[dict]) -> list[dict]:
    filtered = []
    for pair in tqdm(qa_pairs, desc="Filtering QA pairs"):
        try:
            ctx = pair["context"][:1500]
            grounded = generate([{"role": "user", "content": GROUNDEDNESS_PROMPT.format(
                context=ctx, question=pair["question"], answer=pair["answer"])}], max_tokens=50, temperature=0.0)
            relevance = generate([{"role": "user", "content": RELEVANCE_PROMPT.format(
                context=ctx, question=pair["question"])}], max_tokens=20, temperature=0.0)
            standalone = generate([{"role": "user", "content": STANDALONE_PROMPT.format(
                question=pair["question"])}], max_tokens=20, temperature=0.0)

            g_score = int(re.search(r'(\d+)', grounded).group(1)) if re.search(r'(\d+)', grounded) else 1
            r_score = int(re.search(r'(\d+)', relevance).group(1)) if re.search(r'(\d+)', relevance) else 1
            s_score = int(re.search(r'(\d+)', standalone).group(1)) if re.search(r'(\d+)', standalone) else 1

            if g_score >= 4 and r_score >= 4 and s_score >= 4:
                pair["groundedness"] = g_score; pair["relevance"] = r_score; pair["standalone"] = s_score
                filtered.append(pair)
        except Exception:
            continue
    return filtered

print("Synthetic QA generation functions ready.")


### 21c. Configuration Sweep & LLM-as-a-Judge


In [ ]:
def run_evaluation_sweep(pipeline, eval_dataset: list[dict], configs: list[dict]) -> dict:
    results = {}
    for cfg in configs:
        cfg_name = "_".join(f"{k}={v}" for k, v in cfg.items())
        print(f"\nRunning config: {cfg_name}")

        if "chunk_size" in cfg:
            CONFIG["CHUNK_SIZE"] = cfg["chunk_size"]

        outputs = []
        for example in tqdm(eval_dataset, desc=f"Evaluating {cfg_name}"):
            try:
                q_result = pipeline.query(example["question"])
                outputs.append({
                    "question": example["question"],
                    "true_answer": example["answer"],
                    "generated_answer": q_result["answer"],
                    "source_file": example.get("source_file", ""),
                    "config": cfg,
                })
            except Exception as e:
                print(f"Error: {e}")

        results[cfg_name] = outputs
    return results

def evaluate_answers(outputs: list[dict]) -> list[dict]:
    for item in tqdm(outputs, desc="Scoring answers"):
        try:
            prompt = EVAL_PROMPT.format(
                question=item["question"],
                reference=item["true_answer"],
                generated=item["generated_answer"][:500],
            )
            resp = generate([{"role": "user", "content": prompt}], max_tokens=100, temperature=0.0)
            score_match = re.search(r'Score:\s*(\d+)', resp)
            item["eval_score"] = int(score_match.group(1)) / 5.0 if score_match else 0.0
        except Exception:
            item["eval_score"] = 0.0
    return outputs

def compute_metrics(outputs: list[dict]) -> dict:
    scores = [o.get("eval_score", 0) for o in outputs if o.get("eval_score") is not None]
    if not scores: return {"avg_score": 0, "count": 0, "pass_rate": 0}
    return {
        "avg_score": sum(scores) / len(scores),
        "count": len(scores),
        "pass_rate": sum(1 for s in scores if s >= 0.6) / len(scores),
        "max_score": max(scores),
        "min_score": min(scores),
    }

print("Evaluation sweep functions ready.")


## 22. End-to-End Demo


In [ ]:
SAMPLE_TEXT = """# Acme Corporation - Annual Report 2024

## Company Overview
Acme Corporation, founded in 2010 by Jane Smith and John Doe, is a technology company specializing in AI, cloud computing, and cybersecurity. Headquartered in San Francisco, the company employs over 5,000 people across 12 countries.

## Financial Performance
FY 2024 revenue: $2.5 billion (35% YoY growth). Net profit: $420 million (16.8% margin). R&D spend: $380 million.

## Key Projects
- Project Titan: AI platform launched Q3 2024, 10M requests/day, 99.9% uptime, 2,000+ enterprise customers
- Project Nebula: Cloud security framework with CyberGuard Inc., SOC 2 Type II certified June 2024, adopted by 500+ orgs

## Leadership
- Jane Smith: CEO (co-founder), PhD CS from Stanford
- John Doe: CTO (co-founder), former Google AI research lead
- Sarah Johnson: CFO, 20 yrs at Deloitte

## Offices
San Francisco (HQ), London, Bangalore, Tokyo, Berlin"""

sample_file = DATA_DIR / "uploads" / "acme_report_2024.md"
sample_file.parent.mkdir(parents=True, exist_ok=True)
sample_file.write_text(SAMPLE_TEXT, encoding="utf-8")
print(f"Sample document created: {sample_file}")

pipeline = RAGPipeline()
pipeline.load_indices()
result = pipeline.ingest(sample_file, "acme_report_2024.md")
print(f"Ingested: {result['status']} | {result['chunk_count']} chunks | Docs: {pipeline.vector_store.total_documents}")

test_questions = [
    "What is Acme Corporation and who founded it?",
    "Extract all project names and their descriptions",
    "How many offices does Acme have?",
    "Summarize the financial performance",
]

for q in test_questions:
    print(f"\n{'='*60}\nQ: {q}\n{'-'*60}")
    r = pipeline.query(q)
    print(f"Intent: {r.get('intent', 'N/A')}")
    print(f"Answer:\n{r['answer'][:500]}")
    v = r.get('verification', {})
    print(f"Grounded: {v.get('is_grounded')} | Confidence: {v.get('confidence', 0):.3f}")
    if r.get('citations'): print(f"Citations: {len(r['citations'])} sources")


## 23. Run Full Evaluation & Visualization


In [ ]:
print("=" * 60)
print("RAG EVALUATION PIPELINE")
print("=" * 60)

print("\nStep 1: Generating synthetic QA dataset...")
all_chunks = list(pipeline.vector_store.metadata.values())
if all_chunks:
    qa_pairs = generate_qa_pairs(all_chunks, num_pairs=10)
    print(f"Generated {len(qa_pairs)} raw QA pairs")

    print("\nStep 2: Filtering QA pairs...")
    eval_dataset = filter_qa_pairs(qa_pairs)
    print(f"Filtered to {len(eval_dataset)} high-quality QA pairs")
else:
    print("No chunks found. Run the demo cell first to ingest documents.")
    eval_dataset = []

if eval_dataset:
    print("\nStep 3: Running configuration sweep...")
    configs = [
        {"chunk_size": 1024, "rerank": True},
        {"chunk_size": 512, "rerank": True},
    ]
    sweep_results = run_evaluation_sweep(pipeline, eval_dataset, configs)

    print("\nStep 4: Evaluating answers...")
    all_metrics = {}
    for cfg_name, outputs in sweep_results.items():
        scored = evaluate_answers(outputs)
        metrics = compute_metrics(scored)
        all_metrics[cfg_name] = metrics
        print(f"  {cfg_name}: avg_score={metrics['avg_score']:.3f}, pass_rate={metrics['pass_rate']:.2%}")

    print("\nStep 5: Results summary")
    print("-" * 60)
    for cfg_name, metrics in all_metrics.items():
        print(f"{cfg_name}:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
        print()

    if plt:
        names = list(all_metrics.keys())
        scores = [all_metrics[n]["avg_score"] for n in names]
        plt.figure(figsize=(10, 5))
        bars = plt.bar(names, scores, color=["#2ecc71", "#3498db"])
        for bar, score in zip(bars, scores):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{score:.2%}", ha='center', va='bottom', fontsize=11)
        plt.title("RAG Configuration Comparison (Answer Correctness)")
        plt.ylabel("Average Score")
        plt.ylim(0, 1.1)
        plt.xticks(rotation=15, ha='right')
        plt.tight_layout()
        plt.savefig(EVAL_DIR / "rag_evaluation_results.png", dpi=150)
        plt.show()
        print(f"\nPlot saved to {EVAL_DIR / 'rag_evaluation_results.png'}")
else:
    print("\nNo evaluation dataset. Skipping sweep.")
